## Merge các file dữ liệu thô

In [1]:
# Merge đơn giản: lấy cột của file CSV đầu tiên, gộp tất cả file trong ../data/extracted
# Ghi ra ../data/unified/raw_merged.csv

from pathlib import Path
import pandas as pd
import re

IN_DIR  = Path("../data/extracted")            # đổi thành ../data/preprocessed nếu cần
OUT_CSV = Path("../data/unified/raw_merged.csv")  # đổi tên nếu bạn gộp preprocessed

IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

def clean_and_dedup_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Loại BOM, trim khoảng trắng, gom cột trùng (giữ cột đầu tiên)
    new_cols = []
    keep_idx = []
    seen = set()
    for idx, c in enumerate(df.columns):
        col = str(c).replace("\ufeff", "")       # bỏ BOM
        col = re.sub(r"\s+", " ", col).strip()   # gọn khoảng trắng
        key = col.lower()                        # so trùng không phân biệt hoa/thường
        if key not in seen:
            seen.add(key)
            new_cols.append(col)
            keep_idx.append(idx)
        else:
            # bỏ cột trùng, giữ bản đầu tiên
            continue
    df = df.iloc[:, keep_idx]
    df.columns = new_cols
    return df

def read_relaxed_csv(p: Path) -> pd.DataFrame:
    # Đọc “khoan dung”: ưu tiên utf-8-sig, bỏ qua byte lỗi để tránh UnicodeDecodeError
    return pd.read_csv(
        p,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        encoding_errors="ignore",
        on_bad_lines="skip"
    )

paths = sorted(IN_DIR.glob("*.csv"))
if not paths:
    print(f"Không tìm thấy CSV trong {IN_DIR.resolve()}")
else:
    # File đầu tiên xác định bộ cột chuẩn
    df0 = read_relaxed_csv(paths[0])
    df0 = clean_and_dedup_columns(df0)
    base_cols = list(df0.columns)
    frames = [df0[base_cols]]
    print(f"Chuẩn cột dựa trên: {paths[0].name} -> {base_cols}")

    # Các file còn lại: chỉ lấy đúng các cột base_cols (thiếu thì bù cột rỗng)
    for p in paths[1:]:
        dfi = read_relaxed_csv(p)
        dfi = clean_and_dedup_columns(dfi)
        for c in base_cols:
            if c not in dfi.columns:
                dfi[c] = ""
        frames.append(dfi[base_cols])
        print(f"Đã gộp: {p.name} (rows={len(dfi)})")

    merged = pd.concat(frames, ignore_index=True)
    merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Gộp xong {len(paths)} file → {OUT_CSV} ({len(merged)} dòng)")
    try:
        display(merged.head())
    except:
        pass

Chuẩn cột dựa trên: batdongsan_raw.csv -> ['tieu_de', 'gia', 'dia_chi', 'dien_tich_dat', 'phong_ngu', 'phong_tam', 'so_tang', 'phap_ly', 'ngay_dang']
Đã gộp: mogi_raw.csv (rows=2774)
Đã gộp: muaban_raw.csv (rows=2521)
Đã gộp: thuviennhadat_raw.csv (rows=2500)
Gộp xong 4 file → ..\data\unified\raw_merged.csv (10355 dòng)


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,Chính chủ nhờ đăng tin bán nhà hẻm xe hơi 94/8...,"21,5 tỷ","94/8 Đường Trần Khắc Chân, Phường Tân Định, Qu...",32 m²,3 phòng,3 phòng,3 tầng,Sổ đỏ/ Sổ hồng,22/11/2025
1,"Nhà 2 mặt tiền 15 Phan Tôn, P. Tân Định Quận 1","16,5 tỷ","15, Đường Phan Tôn, Phường Đa Kao, Quận 1, Hồ ...","55,3 m²",3 phòng,2 phòng,2 tầng,Sổ đỏ/ Sổ hồng.,26/11/2025
2,Biệt thự góc 2MT hẻm Trần Đình Xu thông Trần H...,82 tỷ,"Đường Trần Đình Xu, Phường Cầu Kho, Quận 1, Hồ...",320 m²,N/A,N/A,2 tầng,Sổ đỏ/ Sổ hồng,24/11/2025
3,"Bán nhà trung tâm q1 hẻm 457 Nguyễn Cảnh Chân,...","17,5 tỷ","Đường Nguyễn Cảnh Chân, Phường Cầu Kho, Quận 1...",66 m²,5 phòng,6 phòng,4 tầng,Sổ đỏ/ Sổ hồng,21/11/2025
4,"Bán gấp giảm nhanh 1,5 tỷ còn 16 tỷ nhà Nguyễn...",16 tỷ,"Đường Nguyễn Thái Bình, Phường Nguyễn Thái Bìn...",73 m²,8 phòng,8 phòng,5 tầng,Sổ đỏ/ Sổ hồng,19/11/2025
